# Beyond Words — Phase 1: GoEmotions Dataset Exploration

**Goal:** inspect GoEmotions, retrieve Google's official Ekman mapping, apply a documented multi-label filtering policy, and create reproducible dataset audit files.

This notebook is designed for both Google Colab and local VS Code/Jupyter.

## Decisions implemented

- Download the official `ekman_mapping.json` from Google Research.
- Keep `neutral` as a separate seventh category.
- GoEmotions is multi-label. Retain only:
  - examples whose only source label is `neutral`; or
  - examples whose non-neutral labels all map to exactly one Ekman category.
- Exclude examples spanning multiple Ekman categories.
- Exclude `neutral` mixed with another label.
- Preserve every row in an audit file, including excluded examples and exclusion reasons.
- Do not treat Ghosh et al.'s `Others` as equivalent to GoEmotions `neutral`.

**Important:** This is a proposed preprocessing policy for this project. Review the counts and label decisions before training. This notebook does not train a model.

## 1. Install dependencies

Run this cell in Colab or in the selected VS Code notebook kernel. In VS Code, you can alternatively install the packages using `pip install -r requirements.txt`.

In [ ]:
pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 42.9 MB/s  0:00:00


In [ ]:
!uv pip install -q -U datasets pandas numpy scikit-learn matplotlib requests pyarrow

## 2. Imports, reproducibility and project paths

In [ ]:
from pathlib import Path
from collections import Counter
import json
import platform
import random
import sys

import numpy as np
import pandas as pd
import requests
from datasets import load_dataset
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import google.colab  # type: ignore
    IN_COLAB = True
    PROJECT_ROOT = Path("/content/Beyond-Words")
except ImportError:
    IN_COLAB = False
    PROJECT_ROOT = Path.cwd()

for relative_path in [
    "data/raw", "data/processed", "data/splits",
    "data/hinglish_custom", "reports", "notebooks"
]:
    (PROJECT_ROOT / relative_path).mkdir(parents=True, exist_ok=True)

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"Running in Colab: {IN_COLAB}")
print(f"Project root: {PROJECT_ROOT.resolve()}")

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Python: 3.13.16
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
Running in Colab: True
Project root: /content/Beyond-Words


## 3. Check compute availability

A GPU is not needed for this dataset audit. We will check whether one is available for later model training.

In [ ]:
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("No CUDA GPU detected. That is fine for Phase 1.")
except ImportError:
    print("PyTorch is not installed in this environment yet.")

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


## 4. Download Google's official Ekman mapping

The mapping is downloaded directly from the Google Research repository and saved locally so the exact mapping used can be audited later.

In [ ]:
EKMAN_MAPPING_URL = (
    "https://raw.githubusercontent.com/google-research/google-research/"
    "master/goemotions/data/ekman_mapping.json"
)

response = requests.get(EKMAN_MAPPING_URL, timeout=30)
response.raise_for_status()
ekman_mapping = response.json()

mapping_path = PROJECT_ROOT / "data/raw/ekman_mapping_official.json"
mapping_path.write_text(
    json.dumps(ekman_mapping, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Mapping saved to:", mapping_path)
display(pd.DataFrame([
    {"ekman_category": category, "goemotions_labels": ", ".join(labels)}
    for category, labels in ekman_mapping.items()
]))

Mapping saved to: /content/Beyond-Words/data/raw/ekman_mapping_official.json


,ekman_category,goemotions_labels
0,anger,"anger, annoyance, disapproval"
1,disgust,disgust
2,fear,"fear, nervousness"
3,joy,"joy, amusement, approval, excitement, gratitud..."
4,sadness,"sadness, disappointment, embarrassment, grief,..."
5,surprise,"surprise, realization, confusion, curiosity"


## 5. Load GoEmotions

We use the `simplified` configuration from Hugging Face Datasets. The notebook inspects the schema rather than assuming label IDs or column names.

In [ ]:
from datasets import load_dataset
go = load_dataset(
    "google-research-datasets/go_emotions",
    "simplified"
)


print(go)
print("Available splits:", list(go.keys()))
print("Training features:")
print(go["train"].features)
print("First training example:")
print(go["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})
Available splits: ['train', 'validation', 'test']
Training features:
{'text': Value('string'), 'labels': List(ClassLabel(names=['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral'])), 'id': Value('string')}
First training example:
{'text': "My favourite food is anything I didn't have to cook myself.", 'labels': [27], 'id': 'eebbqej'}


## 6. Inspect source labels and validate mapping coverage

This cell checks whether every source label is either included in the official six-category Ekman mapping or is `neutral`. If an unexpected label appears, stop and inspect it rather than silently dropping it.

In [ ]:
label_feature = go["train"].features["labels"].feature

if not hasattr(label_feature, "names") or label_feature.names is None:
    raise ValueError(
        "The labels feature does not expose ClassLabel names. "
        "Inspect the dataset schema before proceeding; do not guess label IDs."
    )

goemotion_label_names = list(label_feature.names)

source_to_ekman = {}
for ekman_category, source_labels in ekman_mapping.items():
    for source_label in source_labels:
        if source_label in source_to_ekman:
            raise ValueError(f"Duplicate mapping for source label: {source_label}")
        source_to_ekman[source_label] = ekman_category

neutral_label = "neutral"
unmapped_labels = sorted(
    set(goemotion_label_names) - set(source_to_ekman) - {neutral_label}
)

print("Number of source labels:", len(goemotion_label_names))
print("Source labels:", goemotion_label_names)
print("Ekman categories:", list(ekman_mapping.keys()))
print("Neutral available:", neutral_label in goemotion_label_names)
print("Unmapped labels:", unmapped_labels)

if unmapped_labels:
    raise ValueError(
        "Some GoEmotions labels are not covered by the official mapping or neutral. "
        "Review these labels and update the documented policy before continuing."
    )

Number of source labels: 28
Source labels: ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']
Ekman categories: ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
Neutral available: True
Unmapped labels: []


## 7. Apply the proposed multi-label mapping policy

GoEmotions examples can have more than one label. To make a single-label baseline, this notebook uses a conservative rule:

1. `neutral` alone → `neutral`.
2. One or more non-neutral labels all belonging to one Ekman category → that category.
3. Labels spanning multiple Ekman categories → excluded.
4. `neutral` mixed with another label → excluded.
5. Unknown labels → excluded and recorded.

This avoids arbitrarily choosing one label from a genuinely multi-category example.

In [ ]:
def decode_source_labels(label_ids, label_names):
    decoded = []
    for label_id in label_ids:
        if isinstance(label_id, (int, np.integer)):
            decoded.append(label_names[int(label_id)])
        else:
            decoded.append(str(label_id))
    return decoded


def map_goemotions_example(label_ids, label_names, mapping):
    source_labels = decode_source_labels(label_ids, label_names)
    source_set = set(source_labels)

    unknown = sorted(source_set - set(mapping) - {neutral_label})
    if unknown:
        return source_labels, None, False, "unknown_source_label"

    has_neutral = neutral_label in source_set
    non_neutral = source_set - {neutral_label}

    if has_neutral and not non_neutral:
        return source_labels, "neutral", True, "retained_neutral_only"

    if has_neutral and non_neutral:
        return source_labels, None, False, "neutral_mixed_with_other"

    ekman_targets = sorted({mapping[label] for label in non_neutral})

    if len(ekman_targets) == 1:
        return source_labels, ekman_targets[0], True, "retained_single_ekman_group"

    if len(ekman_targets) == 0:
        return source_labels, None, False, "no_mappable_label"

    return source_labels, None, False, "multiple_ekman_categories"


records = []

for split_name, split_data in go.items():
    for row_index, example in enumerate(split_data):
        source_labels, target, keep, reason = map_goemotions_example(
            example["labels"],
            goemotion_label_names,
            source_to_ekman
        )

        records.append({
            "split": split_name,
            "row_index": row_index,
            "example_id": str(example.get("id", row_index)),
            "text": example["text"],
            "source_labels": "|".join(source_labels),
            "mapped_label": target,
            "keep": keep,
            "mapping_reason": reason,
        })

mapped_df = pd.DataFrame(records)

print(f"Original examples: {len(mapped_df):,}")
print(f"Retained examples: {int(mapped_df['keep'].sum()):,}")
print(f"Excluded examples: {int((~mapped_df['keep']).sum()):,}")

display(
    mapped_df["mapping_reason"]
    .value_counts()
    .rename_axis("mapping_reason")
    .to_frame("examples")
)

Original examples: 54,263
Retained examples: 49,469
Excluded examples: 4,794


,examples
mapping_reason,
retained_single_ekman_group,33448
retained_neutral_only,16021
multiple_ekman_categories,3043
neutral_mixed_with_other,1751


## 8. Inspect retained class counts by split

Class support must always be reported alongside per-class metrics. This table is a data audit, not a model-performance result.

In [ ]:
retained_df = mapped_df.loc[mapped_df["keep"]].copy()

class_counts = (
    retained_df
    .groupby(["split", "mapped_label"])
    .size()
    .rename("support")
    .reset_index()
    .sort_values(["split", "mapped_label"])
)

display(class_counts)

class_pivot = (
    class_counts
    .pivot(index="mapped_label", columns="split", values="support")
    .fillna(0)
    .astype(int)
)

display(class_pivot)

,split,mapped_label,support
0,test,anger,572
1,test,disgust,76
2,test,fear,80
3,test,joy,1863
4,test,neutral,1606
5,test,sadness,283
6,test,surprise,488
7,train,anger,4293
8,train,disgust,498
9,train,fear,541


split,test,train,validation
mapped_label,,,
anger,572,4293,555
disgust,76,498,61
fear,80,541,72
joy,1863,15216,1941
neutral,1606,12823,1592
sadness,283,2326,266
surprise,488,3858,459


In [ ]:
# ============================================================
# Duplicate-text audit and removal for GoEmotions.
# Keeps each normalized text only in its most protected split:
# test > validation > train.
# ============================================================

def normalize_text(s: str) -> str:
    """Lowercase text and collapse whitespace for duplicate detection."""
    if not isinstance(s, str):
        raise TypeError(f"Expected text to be a string, got {type(s).__name__}")
    return " ".join(s.lower().split())


dedup_df = retained_df.copy()
dedup_df["text_norm"] = dedup_df["text"].map(normalize_text)

SPLIT_RANK = {"test": 0, "validation": 1, "train": 2}
dedup_df["split_rank"] = dedup_df["split"].map(SPLIT_RANK)

if dedup_df["split_rank"].isna().any():
    unknown_splits = sorted(dedup_df.loc[dedup_df["split_rank"].isna(), "split"].unique())
    raise ValueError(f"Unexpected split names: {unknown_splits}")

# Step 1: remove cross-split copies, preserving test, then validation, then train.
best_rank = dedup_df.groupby("text_norm")["split_rank"].transform("min")
cross_split_removed = dedup_df.loc[dedup_df["split_rank"] != best_rank].copy()
cross_split_removed["dedup_reason"] = "cross_split_copy"
dedup_df = dedup_df.loc[dedup_df["split_rank"] == best_rank].copy()

# Step 2: within a split, drop every copy of text with conflicting labels.
n_labels = dedup_df.groupby(["split", "text_norm"])["mapped_label"].transform("nunique")
conflict_removed = dedup_df.loc[n_labels > 1].copy()
conflict_removed["dedup_reason"] = "conflicting_labels_within_split"
dedup_df = dedup_df.loc[n_labels == 1].copy()

# Step 3: for same-text/same-label repeats within a split, keep the first row.
is_repeat = dedup_df.duplicated(subset=["split", "text_norm"], keep="first")
same_label_removed = dedup_df.loc[is_repeat].copy()
same_label_removed["dedup_reason"] = "same_label_repeat"
dedup_df = dedup_df.loc[~is_repeat].copy()

removed_df = pd.concat(
    [cross_split_removed, conflict_removed, same_label_removed],
    ignore_index=True
)

# Sanity checks: no normalized text occurs across splits or more than once per split.
assert (dedup_df.groupby("text_norm")["split"].nunique() == 1).all()
assert not dedup_df.duplicated(subset=["split", "text_norm"]).any()
assert len(retained_df) == len(dedup_df) + len(removed_df)

print(f"Retained before dedup: {len(retained_df):,}")
print(f"Removed: {len(removed_df):,}")
display(removed_df["dedup_reason"].value_counts().to_frame("examples"))
print("Removed by split:", removed_df["split"].value_counts().to_dict())
print(f"Retained after dedup: {len(dedup_df):,}")

dedup_counts = (
    dedup_df.pivot_table(
        index="mapped_label",
        columns="split",
        values="example_id",
        aggfunc="count"
    )
    .reindex(
        index=list(ekman_mapping.keys()) + [neutral_label],
        columns=["train", "validation", "test"]
    )
    .fillna(0)
    .astype(int)
)
display(dedup_counts)

dedup_df.drop(columns=["split_rank"]).to_csv(
    PROJECT_ROOT / "data/processed/goemotions_ekman_dedup.csv", index=False
)
removed_df.drop(columns=["split_rank"]).to_csv(
    PROJECT_ROOT / "reports/goemotions_dedup_removed.csv", index=False
)
dedup_counts.to_csv(
    PROJECT_ROOT / "reports/goemotions_class_counts_dedup.csv"
)

Retained before dedup: 49,469
Removed: 320


,examples
dedup_reason,
cross_split_copy,161
same_label_repeat,105
conflicting_labels_within_split,54


Removed by split: {'train': 298, 'validation': 16, 'test': 6}
Retained after dedup: 49,149


split,train,validation,test
mapped_label,,,
anger,4280,555,572
disgust,493,61,76
fear,537,72,80
joy,15032,1928,1858
sadness,2313,266,283
surprise,3840,457,488
neutral,12762,1591,1605


In [ ]:
# Ensure output directories exist.
(PROJECT_ROOT / "data/processed").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "reports").mkdir(parents=True, exist_ok=True)

## 9. Save audit outputs

The full audit CSV contains both retained and excluded examples. The retained CSV is the candidate single-label dataset. Do not use it for training until you have reviewed the exclusions and label distribution.

In [ ]:
audit_path = PROJECT_ROOT / "data/processed/goemotions_mapping_audit.csv"
retained_path = PROJECT_ROOT / "data/processed/goemotions_ekman_retained.csv"
counts_path = PROJECT_ROOT / "reports/goemotions_class_counts.csv"
summary_path = PROJECT_ROOT / "reports/goemotions_mapping_summary.json"

mapped_df.to_csv(audit_path, index=False)
retained_df.to_csv(retained_path, index=False)
class_counts.to_csv(counts_path, index=False)

summary = {
    "dataset": "go_emotions",
    "configuration": "simplified",
    "mapping_source": EKMAN_MAPPING_URL,
    "seed": SEED,
    "original_examples": int(len(mapped_df)),
    "retained_examples": int(mapped_df["keep"].sum()),
    "excluded_examples": int((~mapped_df["keep"]).sum()),
    "retained_label_counts": {
        str(label): int(count)
        for label, count in retained_df["mapped_label"].value_counts().items()
    },
    "exclusion_reason_counts": {
        str(reason): int(count)
        for reason, count in (
            mapped_df.loc[~mapped_df["keep"], "mapping_reason"].value_counts().items()
        )
    },
    "source_label_names": goemotion_label_names,
    "official_ekman_mapping": ekman_mapping,
    "neutral_policy": (
        "Keep neutral as a separate seventh class; do not merge it with "
        "Ghosh et al.'s Others category."
    ),
    "multilabel_policy": (
        "Retain neutral-only examples or non-neutral examples whose source labels "
        "all map to exactly one Ekman category. Exclude mixed-category examples "
        "and neutral mixed with another label."
    ),
}

summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Saved audit:", audit_path)
print("Saved retained data:", retained_path)
print("Saved class counts:", counts_path)
print("Saved summary:", summary_path)

Saved audit: /content/Beyond-Words/data/processed/goemotions_mapping_audit.csv
Saved retained data: /content/Beyond-Words/data/processed/goemotions_ekman_retained.csv
Saved class counts: /content/Beyond-Words/reports/goemotions_class_counts.csv
Saved summary: /content/Beyond-Words/reports/goemotions_mapping_summary.json


In [ ]:
# Record deduplication details and software versions in the summary JSON.
import datasets
import sklearn

summary["dedup"] = {
    "rule": (
        "Keep a text only in its most protected split (test > validation > train); "
        "drop all copies with conflicting labels inside a split; keep one copy otherwise."
    ),
    "retained_before": int(len(retained_df)),
    "retained_after": int(len(dedup_df)),
    "removed_total": int(len(removed_df)),
    "removed_by_reason": {
        str(k): int(v)
        for k, v in removed_df["dedup_reason"].value_counts().items()
    },
    "removed_by_split": {
        str(k): int(v)
        for k, v in removed_df["split"].value_counts().items()
    },
}

summary["versions"] = {
    "python": platform.python_version(),
    "datasets": datasets.__version__,
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
}

summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)
print(json.dumps(summary["versions"], indent=2))
print("Updated summary:", summary_path)

{
  "python": "3.13.16",
  "datasets": "5.2.0",
  "pandas": "3.1.1",
  "numpy": "2.5.4",
  "scikit_learn": "1.9.1"
}
Updated summary: /content/Beyond-Words/reports/goemotions_mapping_summary.json


## 10. Sanity checks

In [ ]:
assert mapped_df["example_id"].notna().all()
assert retained_df["mapped_label"].notna().all()
assert set(retained_df["mapped_label"].unique()).issubset(
    set(ekman_mapping.keys()) | {neutral_label}
)
assert not retained_df.duplicated(subset=["split", "example_id"]).any()

print("All sanity checks passed.")
print("Retained labels:", sorted(retained_df["mapped_label"].unique()))
print("Next: manually review the audit CSV, exclusion counts and class support.")

assert "text_norm" in dedup_df.columns
assert not dedup_df.duplicated(subset=["split", "text_norm"]).any()
assert (dedup_df.groupby("text_norm")["split"].nunique() == 1).all()
assert len(dedup_df) == len(retained_df) - len(removed_df)
print("Deduplication sanity checks passed.")

All sanity checks passed.
Retained labels: ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
Next: manually review the audit CSV, exclusion counts and class support.
Deduplication sanity checks passed.


## 11. Dataset inventory — candidates requiring separate audit

Do not automatically combine these datasets. Before use, verify the original release, license and terms, language/script composition, label definitions, annotation provenance, split provenance and possible overlap with the custom test set.

| Candidate | Possible role | Audit questions |
|---|---|---|
| Ghosh et al. emotion-annotated SentiMix | Hinglish emotion adaptation | Is the original annotated release accessible? What are its terms and exact split files? |
| HinEmo-9 (2026) | Candidate emotion data | Separate human-gold and model-derived tiers; verify Romanized subset and text-use terms |
| GPT-labelled Hinglish emotion data | Weakly supervised training only, if appropriate | Verify provenance, license and generated-label limitations |
| Hinglish sarcasm corpora | Sarcasm adaptation | Verify annotation source, imbalance, license and split leakage |
| Custom 300–500 examples | Independent evaluation | Keep untouched by training, tuning, threshold selection and calibration fitting |

Potential resource links:
- GoEmotions mapping: https://github.com/google-research/google-research/blob/master/goemotions/data/ekman_mapping.json
- HinEmo repository: https://github.com/im-rakeshsahani/hinemo

**Do not assume that a dataset's annotations license also grants rights to redistribute its underlying social-media text.**

## 12. What to share before Phase 2

Please share these outputs:
1. `reports/goemotions_mapping_summary.json` (now includes deduplication results and library versions)
2. `reports/goemotions_class_counts_dedup.csv`
3. The printed dataset schema, before/after dedup counts, removal reasons and any errors or warnings.

We will verify the retained/excluded counts, deduplication results and mapping policy before training the first baseline.